# Final MDT/OST Ranker Comparison — CatBoost vs LightGBM

This notebook performs the controlled comparison required before freezing the official ranking model.

**Contract**
- same final MDT/OST datasets;
- same case-level train/validation/test splits;
- same model features;
- same relevance label;
- same seeds: `21, 42, 84, 126, 168`;
- same evaluation metrics;
- deterministic hard filters are applied before ML;
- model-family selection is based on **validation**, while **test** is reserved for final confirmation.

Expected final dataset hashes:
- MDT: `a6dbcb1ae8c446f626a05d1f8393500a8ee77292770baf6e6ce10dc5824b273c`
- OST: `28380ba8e4ae5d988da834b5d74bce6bd3062d2a948cdece3710227ae51fd2b2`


In [ ]:
import sys, subprocess, importlib.util

required = ["lightgbm", "catboost", "tabulate"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import lightgbm, catboost, pandas as pd
print("Python   :", sys.version.split()[0])
print("LightGBM :", lightgbm.__version__)
print("CatBoost :", catboost.__version__)
print("Pandas   :", pd.__version__)


In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout[:4000])


In [ ]:
from pathlib import Path
import shutil, zipfile

WORK = Path("/kaggle/working")
TRAIN_DIR = WORK / "training"
OUT_ROOT = WORK / "ranker_comparison_final"

for path in [TRAIN_DIR, OUT_ROOT]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

matches = sorted(Path("/kaggle/input").rglob("lustre_ranker_training_data_kaggle.zip"))
if not matches:
    raise FileNotFoundError(
        "lustre_ranker_training_data_kaggle.zip not found under /kaggle/input. "
        "Add the FINAL training ZIP generated after the OST unit correction."
    )

training_zip = matches[0]
print("Using training package:", training_zip)

with zipfile.ZipFile(training_zip) as zf:
    zf.extractall(TRAIN_DIR)

print("Training directory files:")
for p in sorted(TRAIN_DIR.iterdir()):
    print(" -", p.name, p.stat().st_size)


In [ ]:
from pathlib import Path

trainer_path = Path("/kaggle/working/train_ranker_comparison_final.py")
trainer_path.write_text('from __future__ import annotations\n\nimport argparse\nimport gzip\nimport hashlib\nimport json\nimport math\nimport platform\nimport sys\nimport time\nfrom pathlib import Path\nfrom statistics import mean, pstdev\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\n\nimport lightgbm as lgb\nfrom catboost import CatBoostRanker, Pool\n\n\nDEFAULT_SEEDS = [21, 42, 84, 126, 168]\nDEFAULT_ROLES = ["mdt", "ost"]\nDEFAULT_MODELS = ["catboost", "lightgbm"]\n\nIDENTIFIER_FALLBACK = [\n    "split",\n    "case_id",\n    "drive_id",\n    "drive_name",\n    "manufacturer",\n    "series",\n]\n\nLABEL_FALLBACK = [\n    "group_size",\n    "teacher_rank",\n    "teacher_score",\n    "relevance_grade",\n    "is_teacher_top1",\n    "is_teacher_top5",\n    "is_teacher_top10",\n]\n\nLIGHTGBM_BASE_PARAMS: dict[str, Any] = {\n    "objective": "lambdarank",\n    "metric": "ndcg",\n    "boosting_type": "gbdt",\n    "n_estimators": 1800,\n    "learning_rate": 0.045,\n    "max_depth": 8,\n    "num_leaves": 255,\n    "min_child_samples": 30,\n    "reg_alpha": 0.0,\n    "reg_lambda": 6.0,\n    "subsample": 0.9,\n    "subsample_freq": 1,\n    "colsample_bytree": 0.9,\n    "label_gain": [0, 1, 3, 7, 15],\n    "n_jobs": -1,\n    "verbosity": -1,\n    "importance_type": "gain",\n}\n\nCATBOOST_BASE_PARAMS: dict[str, Any] = {\n    "loss_function": "YetiRankPairwise",\n    "eval_metric": "NDCG:top=10",\n    "iterations": 1800,\n    "learning_rate": 0.045,\n    "depth": 8,\n    "l2_leaf_reg": 6.0,\n    "random_strength": 0.5,\n    "bootstrap_type": "Bernoulli",\n    "subsample": 0.9,\n    "verbose": False,\n    "allow_writing_files": False,\n}\n\nEXPECTED_FINAL_DATASET_SHA256_UNCOMPRESSED = {\n    "mdt": "a6dbcb1ae8c446f626a05d1f8393500a8ee77292770baf6e6ce10dc5824b273c",\n    "ost": "28380ba8e4ae5d988da834b5d74bce6bd3062d2a948cdece3710227ae51fd2b2",\n}\n\nEXPECTED_FINAL_ROWS = {\n    "mdt": 188_412,\n    "ost": 116_572,\n}\n\n\nclass ComparisonTrainingError(RuntimeError):\n    """Erreur de contrat ou d\'entraînement de la comparaison finale."""\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description=(\n            "Comparaison finale CatBoostRanker vs LightGBM LGBMRanker sur les "\n            "mêmes datasets, splits, features, seeds et métriques MDT/OST."\n        )\n    )\n    parser.add_argument(\n        "--training-dir",\n        type=Path,\n        default=Path("lustre_architecture_generator/output/training"),\n    )\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("lustre_architecture_generator/artifacts/rankers/comparison_final"),\n    )\n    parser.add_argument(\n        "--evaluation-dir",\n        type=Path,\n        default=Path("lustre_architecture_generator/evaluation/ranking/comparison_final"),\n    )\n    parser.add_argument(\n        "--roles",\n        nargs="+",\n        choices=DEFAULT_ROLES,\n        default=DEFAULT_ROLES,\n    )\n    parser.add_argument(\n        "--models",\n        nargs="+",\n        choices=DEFAULT_MODELS,\n        default=DEFAULT_MODELS,\n    )\n    parser.add_argument(\n        "--seeds",\n        nargs="+",\n        type=int,\n        default=DEFAULT_SEEDS,\n    )\n    parser.add_argument(\n        "--device-type",\n        choices=["gpu", "cpu"],\n        default="gpu",\n    )\n    parser.add_argument(\n        "--n-estimators",\n        type=int,\n        default=1800,\n    )\n    parser.add_argument(\n        "--early-stopping-rounds",\n        type=int,\n        default=150,\n    )\n    parser.add_argument(\n        "--max-cases",\n        type=int,\n        default=None,\n        help="Smoke-test uniquement. Ne pas utiliser pour la campagne finale.",\n    )\n    parser.add_argument(\n        "--skip-final-hash-check",\n        action="store_true",\n        help="Autorisé uniquement pour smoke-tests sur des datasets historiques.",\n    )\n    return parser.parse_args()\n\n\ndef sha256_file(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef sha256_uncompressed_gzip(path: Path) -> str:\n    h = hashlib.sha256()\n    with gzip.open(path, "rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef save_json(path: Path, value: Any) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(\n        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False),\n        encoding="utf-8",\n    )\n\n\ndef load_manifest(training_dir: Path) -> dict[str, Any]:\n    path = training_dir / "training_dataset_manifest.json"\n    if not path.exists():\n        raise FileNotFoundError(f"Manifest introuvable: {path}")\n    value = json.loads(path.read_text(encoding="utf-8"))\n    if not isinstance(value, dict):\n        raise ComparisonTrainingError("training_dataset_manifest.json invalide.")\n    return value\n\n\ndef load_role_data(\n    training_dir: Path,\n    manifest: dict[str, Any],\n    role: str,\n    max_cases: int | None,\n    enforce_final_hash: bool,\n) -> tuple[pd.DataFrame, list[str], list[str], list[str], dict[str, Any]]:\n    if role not in manifest or not isinstance(manifest[role], dict):\n        raise ComparisonTrainingError(f"Manifest sans section {role!r}.")\n\n    meta = manifest[role]\n    dataset_path = training_dir / str(meta["file"])\n    if not dataset_path.exists():\n        raise FileNotFoundError(f"Dataset {role} introuvable: {dataset_path}")\n\n    compressed_sha = sha256_file(dataset_path)\n    uncompressed_sha = sha256_uncompressed_gzip(dataset_path)\n\n    if enforce_final_hash:\n        expected_sha = EXPECTED_FINAL_DATASET_SHA256_UNCOMPRESSED[role]\n        if uncompressed_sha != expected_sha:\n            raise ComparisonTrainingError(\n                f"{role}: mauvais dataset final. SHA256 obtenu={uncompressed_sha}, "\n                f"attendu={expected_sha}."\n            )\n\n    df = pd.read_csv(dataset_path, compression="gzip", low_memory=False)\n    full_row_count = int(len(df))\n\n    if enforce_final_hash and full_row_count != EXPECTED_FINAL_ROWS[role]:\n        raise ComparisonTrainingError(\n            f"{role}: row_count={full_row_count}, attendu={EXPECTED_FINAL_ROWS[role]}."\n        )\n\n    if max_cases is not None:\n        all_ids = sorted(df["case_id"].astype(str).unique())\n        keep_ids = all_ids[:max_cases]\n        df = df[df["case_id"].astype(str).isin(keep_ids)].copy()\n\n    identifier_columns = list(meta.get("identifier_columns", IDENTIFIER_FALLBACK))\n    categorical_features = list(meta["categorical_features"])\n    numeric_features = list(meta["numeric_features"])\n    feature_columns = list(meta["model_feature_columns"])\n    label_columns = list(meta.get("label_columns", LABEL_FALLBACK))\n\n    expected_columns = set(\n        identifier_columns + categorical_features + numeric_features + label_columns\n    )\n    missing = sorted(expected_columns - set(df.columns))\n    if missing:\n        raise ComparisonTrainingError(f"{role}: colonnes manquantes: {missing}")\n\n    if feature_columns != categorical_features + numeric_features:\n        raise ComparisonTrainingError(\n            f"{role}: model_feature_columns != categorical_features + numeric_features."\n        )\n\n    forbidden_features = {\n        "teacher_rank",\n        "teacher_score",\n        "relevance_grade",\n        "is_teacher_top1",\n        "is_teacher_top5",\n        "is_teacher_top10",\n        "group_size",\n    }\n    leaked = sorted(set(feature_columns) & forbidden_features)\n    if leaked:\n        raise ComparisonTrainingError(f"{role}: fuite de labels dans les features: {leaked}")\n\n    for col in categorical_features:\n        df[col] = df[col].fillna("NONE").astype(str)\n\n    for col in numeric_features:\n        df[col] = pd.to_numeric(df[col], errors="coerce")\n\n    df["case_id"] = df["case_id"].astype(str)\n    df["drive_id"] = df["drive_id"].astype(str)\n    df["split"] = df["split"].astype(str)\n    df["relevance_grade"] = pd.to_numeric(\n        df["relevance_grade"], errors="raise"\n    ).astype(int)\n    df["teacher_rank"] = pd.to_numeric(df["teacher_rank"], errors="raise")\n\n    valid_splits = {"train", "validation", "test"}\n    observed = set(df["split"].unique())\n    if observed != valid_splits:\n        raise ComparisonTrainingError(\n            f"{role}: splits attendus={sorted(valid_splits)}, obtenus={sorted(observed)}."\n        )\n\n    contract = {\n        "dataset_file": str(dataset_path),\n        "dataset_rows_full": full_row_count,\n        "dataset_rows_used": int(len(df)),\n        "dataset_sha256_compressed": compressed_sha,\n        "dataset_sha256_uncompressed": uncompressed_sha,\n        "feature_columns": feature_columns,\n        "categorical_features": categorical_features,\n        "numeric_features": numeric_features,\n    }\n\n    return df, feature_columns, categorical_features, numeric_features, contract\n\n\ndef sort_for_ranking(df: pd.DataFrame) -> pd.DataFrame:\n    return df.sort_values(\n        ["case_id", "teacher_rank", "drive_id"],\n        kind="stable",\n    ).reset_index(drop=True)\n\n\ndef split_frames(df: pd.DataFrame) -> dict[str, pd.DataFrame]:\n    frames: dict[str, pd.DataFrame] = {}\n    seen: dict[str, str] = {}\n\n    for split in ("train", "validation", "test"):\n        part = sort_for_ranking(df[df["split"] == split].copy())\n        if part.empty:\n            raise ComparisonTrainingError(f"Split vide: {split}")\n        frames[split] = part\n\n        for case_id in part["case_id"].unique():\n            prior = seen.get(case_id)\n            if prior is not None:\n                raise ComparisonTrainingError(\n                    f"case_id {case_id} présent dans {prior} et {split}."\n                )\n            seen[case_id] = split\n\n    return frames\n\n\ndef group_sizes(df: pd.DataFrame) -> list[int]:\n    sizes = df.groupby("case_id", sort=False).size().astype(int).tolist()\n    if sum(sizes) != len(df):\n        raise AssertionError("Somme des tailles de groupes incohérente.")\n    return sizes\n\n\ndef dcg(relevances: list[float], k: int) -> float:\n    values = relevances[:k]\n    return sum(\n        (2.0 ** rel - 1.0) / math.log2(index + 2.0)\n        for index, rel in enumerate(values)\n    )\n\n\ndef ndcg_for_group(group: pd.DataFrame, pred_col: str, k: int) -> float:\n    predicted = group.sort_values(\n        [pred_col, "drive_id"], ascending=[False, True], kind="stable"\n    )\n    rel_pred = predicted["relevance_grade"].astype(float).tolist()\n    rel_ideal = sorted(group["relevance_grade"].astype(float).tolist(), reverse=True)\n    ideal = dcg(rel_ideal, k)\n    return 1.0 if ideal <= 0 else dcg(rel_pred, k) / ideal\n\n\ndef evaluate_predictions(df: pd.DataFrame, pred_col: str) -> dict[str, float]:\n    metrics: dict[str, list[float]] = {\n        "ndcg_at_5": [],\n        "ndcg_at_10": [],\n        "top1_agreement": [],\n        "top3_overlap": [],\n        "recall_at_5": [],\n        "recall_at_10": [],\n        "top10_overlap_count": [],\n        "top10_jaccard": [],\n        "predicted_top1_teacher_rank": [],\n    }\n\n    for _, group in df.groupby("case_id", sort=False):\n        teacher = group.sort_values(["teacher_rank", "drive_id"], kind="stable")\n        predicted = group.sort_values(\n            [pred_col, "drive_id"], ascending=[False, True], kind="stable"\n        )\n\n        metrics["ndcg_at_5"].append(ndcg_for_group(group, pred_col, 5))\n        metrics["ndcg_at_10"].append(ndcg_for_group(group, pred_col, 10))\n\n        teacher_ids = teacher["drive_id"].tolist()\n        pred_ids = predicted["drive_id"].tolist()\n        metrics["top1_agreement"].append(float(pred_ids[0] == teacher_ids[0]))\n\n        for k, name in ((3, "top3_overlap"), (5, "recall_at_5"), (10, "recall_at_10")):\n            t = set(teacher_ids[:k])\n            p = set(pred_ids[:k])\n            metrics[name].append(len(t & p) / max(1, min(k, len(t))))\n\n        t10 = set(teacher_ids[:10])\n        p10 = set(pred_ids[:10])\n        overlap = len(t10 & p10)\n        union = len(t10 | p10)\n        metrics["top10_overlap_count"].append(float(overlap))\n        metrics["top10_jaccard"].append(overlap / union if union else 1.0)\n\n        pred_top_drive = pred_ids[0]\n        teacher_rank = float(\n            group.loc[group["drive_id"] == pred_top_drive, "teacher_rank"].iloc[0]\n        )\n        metrics["predicted_top1_teacher_rank"].append(teacher_rank)\n\n    return {key: mean(values) for key, values in metrics.items()}\n\n\ndef prediction_frame(\n    source: pd.DataFrame,\n    predictions: np.ndarray,\n    model_family: str,\n    role: str,\n    seed: int,\n    split: str,\n) -> pd.DataFrame:\n    result = source[\n        ["case_id", "drive_id", "teacher_rank", "teacher_score", "relevance_grade"]\n    ].copy()\n    result["ml_score"] = np.asarray(predictions, dtype=float)\n    result.insert(0, "split", split)\n    result.insert(0, "seed", seed)\n    result.insert(0, "role", role)\n    result.insert(0, "model_family", model_family)\n    return result\n\n\ndef build_train_category_mappings(\n    train_df: pd.DataFrame,\n    categorical_features: list[str],\n) -> dict[str, list[str]]:\n    mappings: dict[str, list[str]] = {}\n    for col in categorical_features:\n        values = set(train_df[col].fillna("NONE").astype(str))\n        mappings[col] = sorted(values | {"NONE", "__UNKNOWN__"})\n    return mappings\n\n\ndef prepare_lightgbm_frames(\n    frames: dict[str, pd.DataFrame],\n    categorical_features: list[str],\n    category_mappings: dict[str, list[str]],\n) -> dict[str, pd.DataFrame]:\n    prepared: dict[str, pd.DataFrame] = {}\n    for split, frame in frames.items():\n        part = frame.copy()\n        for col in categorical_features:\n            allowed = set(category_mappings[col])\n            values = part[col].fillna("NONE").astype(str)\n            values = values.where(values.isin(allowed), "__UNKNOWN__")\n            part[col] = pd.Categorical(\n                values,\n                categories=category_mappings[col],\n            )\n        prepared[split] = part\n    return prepared\n\n\ndef train_lightgbm_one_seed(\n    role: str,\n    seed: int,\n    frames: dict[str, pd.DataFrame],\n    feature_columns: list[str],\n    categorical_features: list[str],\n    category_mappings: dict[str, list[str]],\n    model_root: Path,\n    device_type: str,\n    n_estimators: int,\n    early_stopping_rounds: int,\n) -> tuple[dict[str, Any], pd.DataFrame, pd.DataFrame]:\n    prepared = prepare_lightgbm_frames(frames, categorical_features, category_mappings)\n    train_df, val_df, test_df = prepared["train"], prepared["validation"], prepared["test"]\n\n    params = dict(LIGHTGBM_BASE_PARAMS)\n    params.update(\n        {\n            "n_estimators": n_estimators,\n            "random_state": seed,\n            "bagging_seed": seed,\n            "feature_fraction_seed": seed,\n            "data_random_seed": seed,\n            "device_type": device_type,\n        }\n    )\n\n    model = lgb.LGBMRanker(**params)\n    history: dict[str, dict[str, list[float]]] = {}\n    callbacks: list[Any] = [lgb.record_evaluation(history)]\n    if early_stopping_rounds > 0:\n        callbacks.append(lgb.early_stopping(early_stopping_rounds, verbose=False))\n\n    start = time.perf_counter()\n    model.fit(\n        train_df[feature_columns],\n        train_df["relevance_grade"],\n        group=group_sizes(train_df),\n        eval_set=[(val_df[feature_columns], val_df["relevance_grade"])],\n        eval_group=[group_sizes(val_df)],\n        eval_at=[5, 10],\n        categorical_feature=categorical_features,\n        callbacks=callbacks,\n    )\n    training_seconds = time.perf_counter() - start\n\n    pred_start = time.perf_counter()\n    val_pred = model.predict(val_df[feature_columns])\n    test_pred = model.predict(test_df[feature_columns])\n    prediction_seconds = time.perf_counter() - pred_start\n\n    val_eval = prediction_frame(val_df, val_pred, "LightGBM", role, seed, "validation")\n    test_eval = prediction_frame(test_df, test_pred, "LightGBM", role, seed, "test")\n    val_metrics = evaluate_predictions(val_eval, "ml_score")\n    test_metrics = evaluate_predictions(test_eval, "ml_score")\n\n    role_dir = model_root / "lightgbm" / role\n    role_dir.mkdir(parents=True, exist_ok=True)\n    model_path = role_dir / f"{role}_lightgbm_ranker_seed_{seed}.txt"\n    model.booster_.save_model(str(model_path), num_iteration=model.best_iteration_)\n    save_json(role_dir / f"{role}_evaluation_history_seed_{seed}.json", history)\n\n    best_iteration = int(model.best_iteration_ or model.n_estimators_)\n    metrics_row: dict[str, Any] = {\n        "model_family": "LightGBM",\n        "role": role,\n        "seed": seed,\n        "device_type": device_type,\n        "best_iteration": best_iteration,\n        "tree_count": int(model.booster_.num_trees()),\n        "training_seconds": training_seconds,\n        "prediction_seconds_validation_and_test": prediction_seconds,\n        "model_size_mib": model_path.stat().st_size / (1024 * 1024),\n    }\n    metrics_row.update({f"validation_{k}": v for k, v in val_metrics.items()})\n    metrics_row.update({f"test_{k}": v for k, v in test_metrics.items()})\n\n    gain = model.booster_.feature_importance(importance_type="gain")\n    split_imp = model.booster_.feature_importance(importance_type="split")\n    importance = pd.DataFrame(\n        {\n            "model_family": "LightGBM",\n            "role": role,\n            "seed": seed,\n            "feature": feature_columns,\n            "importance": gain.astype(float),\n            "secondary_importance": split_imp.astype(float),\n            "importance_type": "gain",\n        }\n    )\n\n    return metrics_row, pd.concat([val_eval, test_eval], ignore_index=True), importance\n\n\ndef make_catboost_pool(\n    df: pd.DataFrame,\n    feature_columns: list[str],\n    categorical_features: list[str],\n) -> Pool:\n    data = df[feature_columns].copy()\n    for col in categorical_features:\n        data[col] = data[col].fillna("NONE").astype(str)\n    return Pool(\n        data=data,\n        label=df["relevance_grade"].astype(float),\n        group_id=df["case_id"].astype(str),\n        cat_features=categorical_features,\n        feature_names=feature_columns,\n    )\n\n\ndef train_catboost_one_seed(\n    role: str,\n    seed: int,\n    frames: dict[str, pd.DataFrame],\n    feature_columns: list[str],\n    categorical_features: list[str],\n    model_root: Path,\n    device_type: str,\n    n_estimators: int,\n    early_stopping_rounds: int,\n) -> tuple[dict[str, Any], pd.DataFrame, pd.DataFrame]:\n    train_df, val_df, test_df = frames["train"], frames["validation"], frames["test"]\n    train_pool = make_catboost_pool(train_df, feature_columns, categorical_features)\n    val_pool = make_catboost_pool(val_df, feature_columns, categorical_features)\n    test_pool = make_catboost_pool(test_df, feature_columns, categorical_features)\n\n    params = dict(CATBOOST_BASE_PARAMS)\n    params.update(\n        {\n            "iterations": n_estimators,\n            "random_seed": seed,\n            "task_type": device_type.upper(),\n            "od_type": "Iter",\n            "od_wait": max(1, early_stopping_rounds),\n            "use_best_model": True,\n        }\n    )\n    if device_type == "gpu":\n        params["devices"] = "0"\n\n    model = CatBoostRanker(**params)\n\n    start = time.perf_counter()\n    model.fit(train_pool, eval_set=val_pool, verbose=False)\n    training_seconds = time.perf_counter() - start\n\n    pred_start = time.perf_counter()\n    val_pred = model.predict(val_pool)\n    test_pred = model.predict(test_pool)\n    prediction_seconds = time.perf_counter() - pred_start\n\n    val_eval = prediction_frame(val_df, val_pred, "CatBoost", role, seed, "validation")\n    test_eval = prediction_frame(test_df, test_pred, "CatBoost", role, seed, "test")\n    val_metrics = evaluate_predictions(val_eval, "ml_score")\n    test_metrics = evaluate_predictions(test_eval, "ml_score")\n\n    role_dir = model_root / "catboost" / role\n    role_dir.mkdir(parents=True, exist_ok=True)\n    model_path = role_dir / f"{role}_catboost_ranker_seed_{seed}.cbm"\n    model.save_model(str(model_path))\n    save_json(\n        role_dir / f"{role}_evaluation_history_seed_{seed}.json",\n        model.get_evals_result(),\n    )\n\n    best_iteration = int(model.get_best_iteration())\n    if best_iteration < 0:\n        best_iteration = int(model.tree_count_ - 1)\n\n    metrics_row: dict[str, Any] = {\n        "model_family": "CatBoost",\n        "role": role,\n        "seed": seed,\n        "device_type": device_type,\n        "best_iteration": best_iteration,\n        "tree_count": int(model.tree_count_),\n        "training_seconds": training_seconds,\n        "prediction_seconds_validation_and_test": prediction_seconds,\n        "model_size_mib": model_path.stat().st_size / (1024 * 1024),\n    }\n    metrics_row.update({f"validation_{k}": v for k, v in val_metrics.items()})\n    metrics_row.update({f"test_{k}": v for k, v in test_metrics.items()})\n\n    importance_values = model.get_feature_importance(train_pool, type="PredictionValuesChange")\n    importance = pd.DataFrame(\n        {\n            "model_family": "CatBoost",\n            "role": role,\n            "seed": seed,\n            "feature": feature_columns,\n            "importance": np.asarray(importance_values, dtype=float),\n            "secondary_importance": np.nan,\n            "importance_type": "PredictionValuesChange",\n        }\n    )\n\n    return metrics_row, pd.concat([val_eval, test_eval], ignore_index=True), importance\n\n\ndef aggregate_summary(run_metrics: pd.DataFrame) -> pd.DataFrame:\n    base_metrics = [\n        "ndcg_at_5",\n        "ndcg_at_10",\n        "top1_agreement",\n        "top3_overlap",\n        "recall_at_5",\n        "recall_at_10",\n        "top10_jaccard",\n        "predicted_top1_teacher_rank",\n    ]\n    metric_columns = (\n        [f"validation_{name}" for name in base_metrics]\n        + [f"test_{name}" for name in base_metrics]\n        + [\n            "training_seconds",\n            "prediction_seconds_validation_and_test",\n            "model_size_mib",\n            "best_iteration",\n        ]\n    )\n    rows: list[dict[str, Any]] = []\n\n    for (model_family, role), group in run_metrics.groupby(\n        ["model_family", "role"], sort=True\n    ):\n        row: dict[str, Any] = {\n            "model_family": model_family,\n            "role": role,\n            "seed_count": int(len(group)),\n            "device_type": str(group["device_type"].iloc[0]),\n        }\n        for col in metric_columns:\n            values = [float(x) for x in group[col].tolist()]\n            row[f"{col}_mean"] = mean(values)\n            row[f"{col}_std"] = pstdev(values) if len(values) > 1 else 0.0\n            row[f"{col}_min"] = min(values)\n            row[f"{col}_max"] = max(values)\n        rows.append(row)\n\n    return pd.DataFrame(rows)\n\n\ndef feature_stability(feature_importance: pd.DataFrame) -> pd.DataFrame:\n    rows: list[dict[str, Any]] = []\n    for (model_family, role, feature), group in feature_importance.groupby(\n        ["model_family", "role", "feature"], sort=True\n    ):\n        values = group["importance"].astype(float).tolist()\n        avg = mean(values)\n        std = pstdev(values) if len(values) > 1 else 0.0\n        rows.append(\n            {\n                "model_family": model_family,\n                "role": role,\n                "feature": feature,\n                "importance_mean": avg,\n                "importance_std": std,\n                "importance_min": min(values),\n                "importance_max": max(values),\n                "coefficient_of_variation": std / avg if avg > 0 else 0.0,\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef model_selection_table(summary: pd.DataFrame) -> pd.DataFrame:\n    # IMPORTANT: sélection uniquement sur VALIDATION. Le test reste une mesure finale\n    # indépendante et n\'intervient pas dans le classement des familles.\n    rows: list[dict[str, Any]] = []\n    for role, group in summary.groupby("role", sort=True):\n        ranked = group.sort_values(\n            [\n                "validation_ndcg_at_5_mean",\n                "validation_top1_agreement_mean",\n                "validation_ndcg_at_10_mean",\n                "validation_ndcg_at_5_std",\n                "prediction_seconds_validation_and_test_mean",\n            ],\n            ascending=[False, False, False, True, True],\n            kind="stable",\n        ).reset_index(drop=True)\n        for index, row in ranked.iterrows():\n            rows.append(\n                {\n                    "role": role,\n                    "comparison_rank_on_validation": index + 1,\n                    "model_family": row["model_family"],\n                    "validation_ndcg_at_5_mean": row["validation_ndcg_at_5_mean"],\n                    "validation_ndcg_at_5_std": row["validation_ndcg_at_5_std"],\n                    "validation_ndcg_at_10_mean": row["validation_ndcg_at_10_mean"],\n                    "validation_top1_agreement_mean": row["validation_top1_agreement_mean"],\n                    "test_ndcg_at_5_mean": row["test_ndcg_at_5_mean"],\n                    "test_ndcg_at_10_mean": row["test_ndcg_at_10_mean"],\n                    "test_top1_agreement_mean": row["test_top1_agreement_mean"],\n                    "test_recall_at_10_mean": row["test_recall_at_10_mean"],\n                    "training_seconds_mean": row["training_seconds_mean"],\n                    "prediction_seconds_validation_and_test_mean": row[\n                        "prediction_seconds_validation_and_test_mean"\n                    ],\n                    "model_size_mib_mean": row["model_size_mib_mean"],\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef family_overall_table(summary: pd.DataFrame) -> pd.DataFrame:\n    rows: list[dict[str, Any]] = []\n    for model_family, group in summary.groupby("model_family", sort=True):\n        rows.append(\n            {\n                "model_family": model_family,\n                "roles": ",".join(sorted(group["role"].astype(str).tolist())),\n                "validation_ndcg_at_5_role_mean": float(group["validation_ndcg_at_5_mean"].mean()),\n                "validation_ndcg_at_10_role_mean": float(group["validation_ndcg_at_10_mean"].mean()),\n                "validation_top1_role_mean": float(group["validation_top1_agreement_mean"].mean()),\n                "test_ndcg_at_5_role_mean": float(group["test_ndcg_at_5_mean"].mean()),\n                "test_ndcg_at_10_role_mean": float(group["test_ndcg_at_10_mean"].mean()),\n                "test_top1_role_mean": float(group["test_top1_agreement_mean"].mean()),\n                "training_seconds_role_sum": float(group["training_seconds_mean"].sum()),\n                "model_size_mib_role_sum": float(group["model_size_mib_mean"].sum()),\n            }\n        )\n    result = pd.DataFrame(rows)\n    return result.sort_values(\n        [\n            "validation_ndcg_at_5_role_mean",\n            "validation_top1_role_mean",\n            "validation_ndcg_at_10_role_mean",\n        ],\n        ascending=[False, False, False],\n        kind="stable",\n    ).reset_index(drop=True)\n\n\ndef build_markdown_report(\n    summary: pd.DataFrame,\n    selection: pd.DataFrame,\n    overall: pd.DataFrame,\n    contract: dict[str, Any],\n) -> str:\n    lines = [\n        "# Final CatBoost vs LightGBM Ranker Comparison",\n        "",\n        "This report compares both ranker families using the exact same final MDT/OST datasets,",\n        "case-level splits, feature schemas, labels, seeds, and evaluation metrics.",\n        "",\n        "## Dataset contract",\n        "",\n    ]\n    for role in ("mdt", "ost"):\n        if role in contract["roles"]:\n            rc = contract["roles"][role]\n            lines.extend(\n                [\n                    f"- **{role.upper()}**: {rc[\'dataset_rows_full\']} rows, "\n                    f"SHA256(uncompressed) `{rc[\'dataset_sha256_uncompressed\']}`",\n                    f"  - cases: {rc[\'split_case_counts\']}",\n                    f"  - rows: {rc[\'split_rows\']}",\n                ]\n            )\n    lines.extend(["", "## Multi-seed summary", "", summary.to_markdown(index=False), ""])\n    lines.extend(["## Role-level decision table", "", selection.to_markdown(index=False), ""])\n    lines.extend(["## Family-level overall table", "", overall.to_markdown(index=False), ""])\n    lines.extend(\n        [\n            "## Decision rule",\n            "",\n            "Model-family selection is based on validation metrics only. The test split is kept "\n            "independent and is reported only as final confirmation. Role-level ranking uses "\n            "validation NDCG@5 first, then validation Top-1 agreement, validation NDCG@10, "\n            "lower seed variability, and inference time. The family-level table summarizes "\n            "both MDT and OST before the official ranker family is frozen.",\n            "",\n        ]\n    )\n    return "\\n".join(lines)\n\n\ndef main() -> None:\n    args = parse_args()\n    if not args.seeds:\n        raise ComparisonTrainingError("Au moins une seed est requise.")\n    if args.n_estimators <= 0:\n        raise ComparisonTrainingError("--n-estimators doit être > 0.")\n    if args.early_stopping_rounds < 0:\n        raise ComparisonTrainingError("--early-stopping-rounds doit être >= 0.")\n\n    enforce_final_hash = not args.skip_final_hash_check and args.max_cases is None\n    manifest = load_manifest(args.training_dir)\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    args.evaluation_dir.mkdir(parents=True, exist_ok=True)\n\n    all_run_metrics: list[dict[str, Any]] = []\n    all_predictions: list[pd.DataFrame] = []\n    all_importance: list[pd.DataFrame] = []\n    role_contract: dict[str, Any] = {}\n\n    print("Final CatBoost vs LightGBM ranking comparison")\n    print("----------------------------------------------")\n    print(f"Roles       : {args.roles}")\n    print(f"Models      : {args.models}")\n    print(f"Seeds       : {args.seeds}")\n    print(f"Device      : {args.device_type}")\n    print(f"Estimators  : {args.n_estimators}")\n    print(f"Early stop  : {args.early_stopping_rounds}")\n    if args.max_cases is not None:\n        print(f"SMOKE TEST  : max_cases={args.max_cases}")\n\n    for role in args.roles:\n        (\n            df,\n            feature_columns,\n            categorical_features,\n            numeric_features,\n            base_contract,\n        ) = load_role_data(\n            args.training_dir,\n            manifest,\n            role,\n            args.max_cases,\n            enforce_final_hash,\n        )\n        frames = split_frames(df)\n        category_mappings = build_train_category_mappings(\n            frames["train"], categorical_features\n        )\n        base_contract["lightgbm_category_mappings_train_only"] = category_mappings\n        base_contract.update(\n            {\n                "split_rows": {k: int(len(v)) for k, v in frames.items()},\n                "split_case_counts": {\n                    k: int(v["case_id"].nunique()) for k, v in frames.items()\n                },\n            }\n        )\n        role_contract[role] = base_contract\n\n        print(\n            f"\\n[{role.upper()}] rows={len(df)} features={len(feature_columns)} "\n            f"cases={df[\'case_id\'].nunique()}"\n        )\n\n        for model_name in args.models:\n            for seed in args.seeds:\n                print(f"  {model_name} seed={seed} ...", end="", flush=True)\n                if model_name == "lightgbm":\n                    metrics_row, predictions, importance = train_lightgbm_one_seed(\n                        role,\n                        seed,\n                        frames,\n                        feature_columns,\n                        categorical_features,\n                        category_mappings,\n                        args.output_dir,\n                        args.device_type,\n                        args.n_estimators,\n                        args.early_stopping_rounds,\n                    )\n                else:\n                    metrics_row, predictions, importance = train_catboost_one_seed(\n                        role,\n                        seed,\n                        frames,\n                        feature_columns,\n                        categorical_features,\n                        args.output_dir,\n                        args.device_type,\n                        args.n_estimators,\n                        args.early_stopping_rounds,\n                    )\n\n                all_run_metrics.append(metrics_row)\n                all_predictions.append(predictions)\n                all_importance.append(importance)\n                print(\n                    f" done | NDCG@5={metrics_row[\'test_ndcg_at_5\']:.6f} "\n                    f"Top1={metrics_row[\'test_top1_agreement\']:.4f} "\n                    f"iter={metrics_row[\'best_iteration\']}"\n                )\n\n    run_metrics = pd.DataFrame(all_run_metrics)\n    predictions = pd.concat(all_predictions, ignore_index=True)\n    importance = pd.concat(all_importance, ignore_index=True)\n    summary = aggregate_summary(run_metrics)\n    stability = feature_stability(importance)\n    selection = model_selection_table(summary)\n    overall = family_overall_table(summary)\n\n    run_metrics.to_csv(args.evaluation_dir / "ranker_comparison_run_metrics.csv", index=False)\n    summary.to_csv(args.evaluation_dir / "ranker_comparison_multiseed_summary.csv", index=False)\n    selection.to_csv(args.evaluation_dir / "ranker_comparison_decision_table.csv", index=False)\n    overall.to_csv(args.evaluation_dir / "ranker_comparison_family_overall.csv", index=False)\n    predictions.to_csv(\n        args.evaluation_dir / "ranker_comparison_predictions.csv.gz",\n        index=False,\n        compression="gzip",\n    )\n    importance.to_csv(\n        args.evaluation_dir / "ranker_comparison_feature_importance.csv",\n        index=False,\n    )\n    stability.to_csv(\n        args.evaluation_dir / "ranker_comparison_feature_stability.csv",\n        index=False,\n    )\n\n    contract = {\n        "schema_version": "1.0",\n        "purpose": "final_catboost_vs_lightgbm_mdt_ost_comparison",\n        "training_timestamp_utc": pd.Timestamp.utcnow().isoformat(),\n        "python_version": sys.version,\n        "platform": platform.platform(),\n        "library_versions": {\n            "pandas": pd.__version__,\n            "numpy": np.__version__,\n            "lightgbm": lgb.__version__,\n            "catboost": __import__("catboost").__version__,\n        },\n        "models": args.models,\n        "roles_requested": args.roles,\n        "seeds": args.seeds,\n        "device_type": args.device_type,\n        "n_estimators": args.n_estimators,\n        "early_stopping_rounds": args.early_stopping_rounds,\n        "ranking_label": "relevance_grade",\n        "group_column": "case_id",\n        "teacher_columns_excluded_from_features": ["teacher_rank", "teacher_score"],\n        "hard_constraints_applied_before_model": True,\n        "split_contract": manifest.get("case_split", {}),\n        "same_splits_for_both_algorithms": True,\n        "same_features_for_both_algorithms": True,\n        "same_seeds_for_both_algorithms": True,\n        "same_metrics_for_both_algorithms": True,\n        "selection_uses_validation_only": True,\n        "test_split_reserved_for_final_confirmation": True,\n        "expected_final_dataset_sha256_uncompressed": EXPECTED_FINAL_DATASET_SHA256_UNCOMPRESSED,\n        "lightgbm_parameters_except_seed_device_estimators": LIGHTGBM_BASE_PARAMS,\n        "catboost_parameters_except_seed_device_iterations": CATBOOST_BASE_PARAMS,\n        "roles": role_contract,\n    }\n    save_json(args.evaluation_dir / "ranker_comparison_contract.json", contract)\n\n    report = build_markdown_report(summary, selection, overall, contract)\n    (args.evaluation_dir / "RANKER_COMPARISON_REPORT.md").write_text(\n        report, encoding="utf-8"\n    )\n\n    print("\\nFinal comparison completed")\n    print("--------------------------")\n    print(summary.to_string(index=False))\n    print("\\nRole-level decision table (validation-based)")\n    print(selection.to_string(index=False))\n    print("\\nFamily-level overall table")\n    print(overall.to_string(index=False))\n    print(f"\\nModels     : {args.output_dir}")\n    print(f"Evaluation : {args.evaluation_dir}")\n    print("STATUS     : VALIDATED")\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
print("Trainer written:", trainer_path)
print("Bytes:", trainer_path.stat().st_size)


In [ ]:
import subprocess, sys

cmd = [
    sys.executable,
    "/kaggle/working/train_ranker_comparison_final.py",
    "--training-dir", "/kaggle/working/training",
    "--output-dir", "/kaggle/working/ranker_comparison_final/models",
    "--evaluation-dir", "/kaggle/working/ranker_comparison_final/evaluation",
    "--roles", "mdt", "ost",
    "--models", "catboost", "lightgbm",
    "--seeds", "21", "42", "84", "126", "168",
    "--device-type", "gpu",
    "--n-estimators", "1800",
    "--early-stopping-rounds", "150",
]

print("Running final comparison...
")
print(" ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
from pathlib import Path
import pandas as pd

EVAL = Path("/kaggle/working/ranker_comparison_final/evaluation")
summary = pd.read_csv(EVAL / "ranker_comparison_multiseed_summary.csv")
decision = pd.read_csv(EVAL / "ranker_comparison_decision_table.csv")
overall = pd.read_csv(EVAL / "ranker_comparison_family_overall.csv")

print("=== ROLE-LEVEL DECISION TABLE (VALIDATION-BASED) ===")
display(decision)
print("=== FAMILY-LEVEL OVERALL TABLE ===")
display(overall)
print("=== FULL MULTI-SEED SUMMARY ===")
display(summary)


In [ ]:
from pathlib import Path
import shutil

root = Path("/kaggle/working/ranker_comparison_final")
zip_base = Path("/kaggle/working/ranker_comparison_final_artifacts")
archive = shutil.make_archive(str(zip_base), "zip", root_dir=root)
print("Final ZIP:", archive)
print("Size MiB:", Path(archive).stat().st_size / (1024 * 1024))


In [ ]:
from IPython.display import FileLink, display
display(FileLink('/kaggle/working/ranker_comparison_final_artifacts.zip'))


## What to return after the run

Download `ranker_comparison_final_artifacts.zip` and send it back for validation.

The next project step is **not Beam Search yet**. We will first:
1. validate this comparison;
2. freeze one official ranker family;
3. integrate its loader/inference;
4. freeze the `Top-K → Architecture` contract;
5. then build the complete architecture layer before Beam Search.
